In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_bronze_framework

In [0]:
import copy

failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


# Specs with limits are valid; wrong limits are rejected
archive = load_source_spec_file("open_meteo_archive")
check("open_meteo_archive spec with limits is valid", not validate_source_spec(archive))
check("open_meteo_archive and open_meteo_forecast share the quota group",
      source_limits(archive)["quota_group"] == source_limits(load_source_spec_file("open_meteo_forecast"))["quota_group"])

bad = copy.deepcopy(archive)
bad["limits"]["per_minute"] = 20000
check("rejects per_minute > per_hour", any("per_minute <= per_hour" in e for e in validate_source_spec(bad)))
bad = copy.deepcopy(archive)
bad["limits"]["cost"]["variables_param"] = "daily"
check("rejects a variables_param that is not an endpoint param",
      any("is not a param" in e for e in validate_source_spec(bad)))
bad = copy.deepcopy(archive)
bad["limits"]["per_second"] = 1
check("rejects unknown limit keys", any("unknown keys" in e for e in validate_source_spec(bad)))

# Cost of an Open-Meteo call: 52 locations x weeks / 2 weeks x variables / 10
limits = source_limits(archive)
hourly = {"hourly": archive["endpoints"][0]["params"]["hourly"]}
week_cost = estimate_call_cost(limits, hourly, date(2026, 11, 2), date(2026, 11, 8), 52)
four_weeks_cost = estimate_call_cost(limits, hourly, date(2026, 10, 12), date(2026, 11, 8), 52)
check(f"a weekly chunk of 52 provinces costs 52 (got {week_cost})", week_cost == 52)
check(f"a 4-week chunk costs 104 (got {four_weeks_cost})", four_weeks_cost == 104)

# Budget decisions (pure logic)
now = datetime(2026, 11, 20, 12, 0, 0)
check("no history -> call", budget_decision([], limits, 52, now)[0] == "call")
check("hour almost used -> defer",
      budget_decision([(now - timedelta(minutes=30), 3990)], limits, 52, now)[0] == "defer")
decision, seconds = budget_decision([(now - timedelta(seconds=10), 470)], limits, 52, now)
check(f"minute almost used -> wait about 50 s (got {decision}, {seconds})", decision == "wait" and 45 <= seconds <= 51)
check("a single call above the per-minute budget -> too_big", budget_decision([], limits, 500, now)[0] == "too_big")
decision, seconds = budget_decision([(now - timedelta(seconds=0.5), 1)], limits, 1, now)
check("minimum gap between calls -> wait", decision == "wait" and seconds <= 1)

In [0]:
check("Retry-After in seconds", abs((block_until_from_retry_after("120") - utc_now()).total_seconds() - 120) < 5)
next_midnight = block_until_from_retry_after(None)
check("no Retry-After -> next UTC midnight", next_midnight.hour == 0 and next_midnight > utc_now())

test_group = "test_governance_group"
unblock_quota_group(test_group)
block_quota_group(test_group, utc_now() + timedelta(hours=1), "test block")
check("block is active", active_block(test_group) is not None)

# A dry run of a blocked quota group makes no HTTP call: every chunk is deferred
blocked_spec = copy.deepcopy(load_source_spec_file("ree"))
blocked_spec["limits"]["quota_group"] = test_group
result = run_source_spec(blocked_spec, window_override=1, dry_run=True)
check("blocked source: every chunk deferred", result["chunks_deferred"] == len(result["results"]) > 0)
check("blocked source: no HTTP call recorded",
      spark.table(api_usage_table).filter(F.col("quota_group") == test_group).count() == 0)

unblock_quota_group(test_group)
check("unblock lifts the block", active_block(test_group) is None)

In [0]:
# Cleanup: delete every record this test created in the production governance tables
test_quota_groups = ["test_governance_group"]

groups_sql = ", ".join(f"'{g}'" for g in test_quota_groups)
for table in [source_blocks_table, api_usage_table]:
    deleted = spark.sql(f"SELECT count(*) AS n FROM {table} WHERE quota_group IN ({groups_sql})").first()["n"]
    if deleted:
        spark.sql(f"DELETE FROM {table} WHERE quota_group IN ({groups_sql})")
    print(f"🧹 {table}: deleted {deleted} test row(s)")

check("no test blocks left", spark.table(source_blocks_table)
      .filter(F.col("quota_group").isin(test_quota_groups)).count() == 0)
check("no test API usage left", spark.table(api_usage_table)
      .filter(F.col("quota_group").isin(test_quota_groups)).count() == 0)

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All API governance checks passed")